# 18-Class S/R/P Flat Algorithm Selector — RF & XGB

This notebook replaces the direct 3-class router with an **18-class flat algorithm selector** over the non-B algorithms.

The model predicts one of 18 algorithms (6 S + 6 R + 6 P). The S/R/P branch is derived afterward from the predicted algorithm name.

## Core methodology

- Active algorithm vocabulary: **18 S/R/P algorithms; B algorithms are excluded from training/prediction**.
- Models: RF and XGB.
- Splits: `strat_y` and `strat_y_K`.
- Existing flat split logic is reused: **split original rows first; expand tied optimal algorithm labels only in training; test rows stay unexpanded**.
- Algorithm correctness accepts any exactly tied fastest algorithm in the 18-way candidate set.
- S/R/P routing metrics are derived from true/predicted algorithms.
- Important-failure and Top-K runtime ratios use the **global minimum runtime across all 24 algorithms** as denominator.
- Important-failure detail workbook contains eight model/split/scope sheets and a test-vs-entire overlap summary.


## Cell 1 — Imports, paths, and experiment definition

This cell defines all paths explicitly and imports the existing pipeline functions.

### Existing functions/modules reused

- `train.py`
  - `load_yaml_config()`
  - `load_and_preprocess_data()`
- `models/flat.py`
  - `fit_random_forest_classifier()`
  - `fit_xgb_classifier()`
- `features.py`
  - canonical 24-algorithm vocabulary;
  - S/R/P/B algorithm groups.
- `compare_wrong_predictions.py`
  - `filter_tolerance_sheet_for_important_failures()`

No existing source file is modified by this notebook.


In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay

REPO_ROOT=Path.cwd()
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))

from restructured_pipeline.src.features import (
    ALL_ALGOS, ROUTER_CLASSB_ALGOS, ROUTER_CLASSS_ALGOS,
    ROUTER_CLASSR_ALGOS, ROUTER_CLASSP_ALGOS,
    load_excel_data, process_kcore_list_to_statistics, post_process_data,
)
from restructured_pipeline.src.train import load_yaml_config, load_and_preprocess_data
from restructured_pipeline.src.models.flat import fit_random_forest_classifier, fit_xgb_classifier
from restructured_pipeline.src.compare_wrong_predictions import filter_tolerance_sheet_for_important_failures

PROJECT_CONFIG=Path("restructured_pipeline/configs/project.yaml")
project=load_yaml_config(PROJECT_CONFIG)
FEATURE_COLS=list(project["features"]["feature_cols"])
S_ALGOS=list(ROUTER_CLASSS_ALGOS); R_ALGOS=list(ROUTER_CLASSR_ALGOS); P_ALGOS=list(ROUTER_CLASSP_ALGOS)
SRP_ALGOS=S_ALGOS+R_ALGOS+P_ALGOS
B_ALGOS=list(ROUTER_CLASSB_ALGOS)
BRANCH_ALGOS={"S":S_ALGOS,"R":R_ALGOS,"P":P_ALGOS}
ALGO_TO_BRANCH={a:b for b,xs in BRANCH_ALGOS.items() for a in xs}
SPLITS=("strat_y","strat_y_K"); MODELS=("RF","XGB")
OUTPUT_DIR=Path("restructured_pipeline/outputs/reports/SRP_18class_flat")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

assert len(SRP_ALGOS)==18 and len(B_ALGOS)==6
assert set(SRP_ALGOS).isdisjoint(B_ALGOS)
assert set(SRP_ALGOS)|set(B_ALGOS)==set(ALL_ALGOS)
print("Active classes:",len(SRP_ALGOS))
for b,xs in BRANCH_ALGOS.items(): print(b,len(xs),xs)


Active classes: 18
S 6 ['kPEX-S-bnb', 'kPEX-oriented-S-bnb', 'UMKP-S-bnb', 'UMKP-oriented-S-bnb', 'UMKP+-S-bnb', 'UMKP+-oriented-S-bnb']
R 6 ['kPEX-R-bnb', 'kPEX-oriented-R-bnb', 'UMKP-R-bnb', 'UMKP-oriented-R-bnb', 'UMKP+-R-bnb', 'UMKP+-oriented-R-bnb']
P 6 ['kPEX-P-bnb', 'kPEX-oriented-P-bnb', 'UMKP-P-bnb', 'UMKP-oriented-P-bnb', 'UMKP+-P-bnb', 'UMKP+-oriented-P-bnb']


## Cell 2 — Load and preprocess the canonical data

**Existing function reused:** `train.py -> load_and_preprocess_data()`.

This is the exact preprocessing entry point used by the training pipeline. It loads the configured raw dataset, assigns the standard weights, generates the k-core/shell statistics, runs post-processing, creates `source_row_index`, and applies the same configured feature sanitization used by normal training.

We call it with `scheme="scheme1"` only to select the standard flat-data preprocessing path. The experiment is **not Scheme 1**: the active model vocabulary is explicitly restricted later to the 18 S/R/P algorithms.

### Paths used

- Project configuration: `restructured_pipeline/configs/project.yaml`
- Model implementation: `restructured_pipeline/src/models/flat.py`
- Training/preprocessing utilities: `restructured_pipeline/src/train.py`
- Outputs: `restructured_pipeline/outputs/reports/SRP_18class_flat/`


In [2]:
# ============================================================
# CANONICAL DATA PREPROCESSING
# Reuse the exact preprocessing entry point from train.py.
# ============================================================

print("Project config:")
print(" ", PROJECT_CONFIG.resolve())

print("Repository root:")
print(" ", REPO_ROOT.resolve())

print("Output directory:")
print(" ", OUTPUT_DIR.resolve())

df = load_and_preprocess_data(
    project,
    scheme="scheme1",
)

# scheme="scheme1" above means standard flat-data preprocessing only.
# The active classifier vocabulary is supplied later as SRP_ALGOS.

required = [
    "dataset",
    "k_value",
    *FEATURE_COLS,
    *ALL_ALGOS,
]

missing = [
    column
    for column in required
    if column not in df.columns
]

if missing:
    raise ValueError(
        "Required columns are missing after train.py preprocessing:\n"
        f"{missing}"
    )

if "source_row_index" not in df.columns:
    raise ValueError(
        "load_and_preprocess_data() did not create source_row_index."
    )

missing_features = int(
    df[FEATURE_COLS]
    .isna()
    .sum()
    .sum()
)

if missing_features:
    raise ValueError(
        "Configured features still contain missing values after "
        "load_and_preprocess_data(): "
        f"{missing_features}"
    )

print()
print("=== PREPROCESSED DATA AUDIT ===")
print("Original cases:", len(df))
print("Configured features:", len(FEATURE_COLS))
print("Active S/R/P algorithms:", len(SRP_ALGOS))
print("Excluded B algorithms:", len(B_ALGOS))
print("Missing feature values:", missing_features)

display(
    df[
        [
            "dataset",
            "k_value",
            "source_row_index",
            *FEATURE_COLS,
        ]
    ].head()
)


Project config:
  D:\Studies\Research\Da_Project\kpex_ML_git\restructured_pipeline\configs\project.yaml
Repository root:
  D:\Studies\Research\Da_Project\kpex_ML_git
Output directory:
  D:\Studies\Research\Da_Project\kpex_ML_git\restructured_pipeline\outputs\reports\SRP_18class_flat

=== PREPROCESSED DATA AUDIT ===
Original cases: 4279
Configured features: 12
Active S/R/P algorithms: 18
Excluded B algorithms: 6
Missing feature values: 0


,dataset,k_value,source_row_index,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize
0,12month1,20,0,20,52328,1585220,30.294,9214,57,19,39,16785.102564,52328,1030,14354.722410
1,130bit,8,1,8,584,6059,10.375,322,12,8,5,564.400000,584,509,28.443628
2,130bit,9,2,9,584,6059,10.375,322,12,8,5,564.400000,584,509,28.443628
3,130bit,10,3,10,583,6051,10.379,321,12,9,4,559.500000,583,509,29.853810
4,144,4,4,4,144319,1072349,7.430,26,9,7,3,140266.333333,144319,132943,5188.210310


## Cell 3 — Train the 18-class flat models using the existing flat pipeline

**Existing functions reused:** `fit_random_forest_classifier()` and `fit_xgb_classifier()` from `restructured_pipeline/src/models/flat.py`.

Both receive:

`algorithm_cols = SRP_ALGOS`

so the active vocabulary is exactly **18 algorithms**.

Internally, the existing flat pipeline calls the existing `data_splits.py` functions for `strat_y` or `strat_y_K`. The established leakage-safe behavior is preserved:

1. construct the canonical target inside the 18-algorithm vocabulary;
2. split **original source rows**;
3. expand tied/near-tied optimal labels **only in training**;
4. keep held-out test rows unexpanded;
5. validate zero train/test source-row overlap;
6. XGB fits its encoder to the complete active 18-class vocabulary.

No notebook-specific splitting or tie-expansion implementation is used.


In [3]:
split_defaults=project["split_rules"]["defaults"]
split_kwargs={
    "weight_col":split_defaults["weight_col"],
    "test_size":split_defaults["test_size"],
    "random_state":split_defaults["random_state"],
    "tie_tolerance":split_defaults["tie_tolerance"],
    "algorithm_cols":SRP_ALGOS,
}
if split_kwargs["weight_col"] not in df.columns:
    df[split_kwargs["weight_col"]]=1.0

trained={}; split_data={}
for split in SPLITS:
    rf,rf_split=fit_random_forest_classifier(df,model_name=f"rf_{split}",split_mode=split,split_kwargs=split_kwargs)
    xgb,le,xgb_split=fit_xgb_classifier(df,model_name=f"xgb_{split}",split_mode=split,split_kwargs=split_kwargs)
    trained[(split,"RF")]=(rf,None); split_data[(split,"RF")]=rf_split
    trained[(split,"XGB")]=(xgb,le); split_data[(split,"XGB")]=xgb_split
    for m in MODELS:
        sd=split_data[(split,m)]
        print(split,m,"classes",sd["num_class"],"train rows",len(sd["X_train"]),"unique train",sd["X_train"].index.nunique(),"test",len(sd["X_test"]))
        assert sd["num_class"]==18
        assert not (set(sd["X_train"].index)&set(sd["X_test"].index))
        assert not sd["X_test"].index.duplicated().any()


Expanded 403 TRAINING source row(s) with near ties using tolerance 0.0%.
Original training rows=3423; expanded training rows=7799.
prepare_train_test_strat_Y: original-row split completed; train/test=3423/856, expanded_train=7799, test_unexpanded=856.
Expanded 403 TRAINING source row(s) with near ties using tolerance 0.0%.
Original training rows=3423; expanded training rows=7799.
prepare_train_test_strat_Y: original-row split completed; train/test=3423/856, expanded_train=7799, test_unexpanded=856.


c:\Users\n1a9m\Anaconda3_\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:08:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


strat_y RF classes 18 train rows 7799 unique train 3423 test 856
strat_y XGB classes 18 train rows 7799 unique train 3423 test 856
prepare_train_test_strat_Y_K: forced 8 row(s) into training before the K-aware split.
prepare_train_test_strat_Y_K: forced class UMKP-P-bnb_k2 (1 row(s)); reason=combined class has fewer than 2 rows; source_row_indices=[1134]
prepare_train_test_strat_Y_K: forced class UMKP-R-bnb_k10 (1 row(s)); reason=combined class has fewer than 2 rows; source_row_indices=[2662]
prepare_train_test_strat_Y_K: forced class UMKP-R-bnb_k15 (1 row(s)); reason=combined class has fewer than 2 rows; source_row_indices=[3804]
prepare_train_test_strat_Y_K: forced class UMKP-R-bnb_k20 (1 row(s)); reason=combined class has fewer than 2 rows; source_row_indices=[741]
prepare_train_test_strat_Y_K: forced class UMKP-R-bnb_k9 (1 row(s)); reason=combined class has fewer than 2 rows; source_row_indices=[649]
prepare_train_test_strat_Y_K: forced class UMKP-S-bnb_k2 (1 row(s)); reason=combin

c:\Users\n1a9m\Anaconda3_\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:09:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


strat_y_K RF classes 18 train rows 7862 unique train 3424 test 855
strat_y_K XGB classes 18 train rows 7862 unique train 3424 test 855


## Cell 4 — Build 18-way probability rankings for test and entire data

The notebook needs a small local helper here because we are not saving/reloading model artifacts. It follows the existing flat-prediction convention:

`(time_ratio, algorithm, runtime, probability)`

The probability ranking contains **all 18 S/R/P algorithms**. Runtime ratios in this ranking use the **24-way global minimum**.


In [4]:
def tied_fastest(row,algos):
    best=min(float(row[a]) for a in algos)
    return [a for a in algos if float(row[a])==best],best

def make_predictions(split,model_name,scope):
    clf,le=trained[(split,model_name)]; sd=split_data[(split,model_name)]
    idx=sd["X_test"].index if scope=="test_data" else df.index
    X=df.loc[idx,FEATURE_COLS]
    raw=clf.predict_proba(X)
    if le is None:
        classes=list(clf.classes_); pred=clf.predict(X)
    else:
        classes=list(le.inverse_transform(np.arange(raw.shape[1])))
        pred=le.inverse_transform(clf.predict(X).astype(int))
    pos={a:i for i,a in enumerate(classes)}
    probs=np.column_stack([raw[:,pos[a]] for a in SRP_ALGOS])
    rows=[]
    for n,i in enumerate(idx):
        src=df.loc[i]; true18,best18=tied_fastest(src,SRP_ALGOS); global24,global_time=tied_fastest(src,list(ALL_ALGOS))
        predicted=str(pred[n]); ranking=[]
        for j,a in enumerate(SRP_ALGOS):
            rt=float(src[a]); pr=float(probs[n,j])
            ranking.append((rt/global_time if global_time else np.nan,a,rt,pr))
        ranking.sort(key=lambda z:-z[3])
        rows.append({
            "source_row_index":i,"dataset":src["dataset"],"k_value":src["k_value"],
            "split_mode":split,"model":model_name,"data_scope":scope,
            "true_fastest_algorithms_18":true18,"canonical_fastest_algo_18":true18[0],
            "predicted_algorithm":predicted,"algorithm_correct":predicted in true18,
            "true_branch":ALGO_TO_BRANCH[true18[0]],"predicted_branch":ALGO_TO_BRANCH[predicted],
            "branch_correct":ALGO_TO_BRANCH[predicted] in {ALGO_TO_BRANCH[a] for a in true18},
            "global_fastest_algorithms_24":global24,"global_fastest_time_24":global_time,
            "predicted_runtime":float(src[predicted]),
            "predicted_time_ratio_24":float(src[predicted])/global_time if global_time else np.nan,
            "all_predictions_sorted_by_prob":ranking,
        })
    return pd.DataFrame(rows)

predictions={(s,m,sc):make_predictions(s,m,sc) for s in SPLITS for m in MODELS for sc in ("test_data","entire_data")}
display(predictions[("strat_y","RF","test_data")].head())


,source_row_index,dataset,k_value,split_mode,model,data_scope,true_fastest_algorithms_18,canonical_fastest_algo_18,predicted_algorithm,algorithm_correct,true_branch,predicted_branch,branch_correct,global_fastest_algorithms_24,global_fastest_time_24,predicted_runtime,predicted_time_ratio_24,all_predictions_sorted_by_prob
0,1168,elec,15,strat_y,RF,test_data,[UMKP-oriented-P-bnb],UMKP-oriented-P-bnb,UMKP-oriented-P-bnb,True,P,P,True,[UMKP-oriented-P-bnb],0.0218,0.0218,1.000000,"[(1.0, UMKP-oriented-P-bnb, 0.0218, 0.53026247..."
1,1259,F2,8,strat_y,RF,test_data,[UMKP-oriented-S-bnb],UMKP-oriented-S-bnb,UMKP-oriented-S-bnb,True,S,S,True,[UMKP-oriented-S-bnb],0.9878,0.9878,1.000000,"[(1.0, UMKP-oriented-S-bnb, 0.9878, 0.31987470..."
2,611,cant,9,strat_y,RF,test_data,[UMKP-oriented-S-bnb],UMKP-oriented-S-bnb,UMKP-oriented-P-bnb,False,S,P,False,[UMKP-oriented-S-bnb],1.7695,1.8308,1.034643,"[(1.0346425543938966, UMKP-oriented-P-bnb, 1.8..."
3,2964,sc-msdoor,20,strat_y,RF,test_data,[UMKP+-oriented-R-bnb],UMKP+-oriented-R-bnb,UMKP+-oriented-R-bnb,True,R,R,True,[UMKP-oriented-B-bnb],1.4652,1.5120,1.031941,"[(1.0319410319410318, UMKP+-oriented-R-bnb, 1...."
4,1266,f855_mat9,5,strat_y,RF,test_data,"[UMKP-oriented-S-bnb, UMKP+-oriented-S-bnb, UM...",UMKP-oriented-S-bnb,UMKP-oriented-R-bnb,True,S,R,True,"[UMKP-oriented-S-bnb, UMKP-oriented-R-bnb, UMK...",0.0017,0.0017,1.000000,"[(1.0, UMKP-oriented-R-bnb, 0.0017, 0.35547036..."


## Cell 5 — Algorithm-selection and derived S/R/P metrics

Two views are reported from the same 18-class model:

1. **Algorithm selection:** exact/tie-aware correctness among the 18 algorithms.
2. **Derived S/R/P routing:** map true and predicted algorithms to S/R/P.

This cell uses sklearn metric functions only for reporting; training/prediction logic is unchanged.


In [5]:
def report_metrics(frame):
    # Algorithm accuracy is tie-aware and therefore taken directly from algorithm_correct.
    alg_acc=float(frame.algorithm_correct.mean())
    yp=frame.predicted_algorithm; yt=frame.canonical_fastest_algo_18
    _,_,alg_f1w,_=precision_recall_fscore_support(yt,yp,labels=SRP_ALGOS,average="weighted",zero_division=0)
    bt=frame.true_branch; bp=frame.predicted_branch
    pm,rm,fm,_=precision_recall_fscore_support(bt,bp,labels=["S","R","P"],average="macro",zero_division=0)
    pw,rw,fw,_=precision_recall_fscore_support(bt,bp,labels=["S","R","P"],average="weighted",zero_division=0)
    p,r,f,sup=precision_recall_fscore_support(bt,bp,labels=["S","R","P"],average=None,zero_division=0)
    z={"Algorithm Accuracy":alg_acc,"Algorithm Weighted F1":alg_f1w,"S/R/P Accuracy":float(frame.branch_correct.mean()),
       "S/R/P Mean Precision":pm,"S/R/P Mean Recall":rm,"S/R/P Mean F1":fm,
       "S/R/P Weighted Precision":pw,"S/R/P Weighted Recall":rw,"S/R/P Weighted F1":fw}
    for j,b in enumerate(["S","R","P"]):
        z.update({f"{b} Precision":p[j],f"{b} Recall":r[j],f"{b} F1":f[j],f"{b} Cases":int(sup[j])})
    return z

metric_rows=[]
for (s,m,sc),f in predictions.items():
    metric_rows.append({"Split":s,"Model":m,"Scope":sc,"Total Cases":len(f),**report_metrics(f)})
metrics=pd.DataFrame(metric_rows); display(metrics.round(4)); metrics.to_csv(OUTPUT_DIR/"metrics.csv",index=False)


,Split,Model,Scope,Total Cases,Algorithm Accuracy,Algorithm Weighted F1,S/R/P Accuracy,S/R/P Mean Precision,S/R/P Mean Recall,S/R/P Mean F1,...,S F1,S Cases,R Precision,R Recall,R F1,R Cases,P Precision,P Recall,P F1,P Cases
0,strat_y,RF,test_data,856,0.4673,0.3624,0.7021,0.5431,0.5363,0.5310,...,0.5110,297,0.2866,0.3659,0.3214,123,0.7152,0.8119,0.7605,436
1,strat_y,RF,entire_data,4279,0.8011,0.7017,0.8841,0.7532,0.7677,0.7485,...,0.7346,1484,0.5244,0.7386,0.6133,612,0.8711,0.9258,0.8976,2183
2,strat_y,XGB,test_data,856,0.4848,0.3883,0.7033,0.5598,0.5525,0.5503,...,0.5426,297,0.3404,0.3902,0.3636,123,0.6996,0.7959,0.7446,436
3,strat_y,XGB,entire_data,4279,0.8619,0.7535,0.9210,0.8238,0.8226,0.8176,...,0.7956,1484,0.7155,0.8137,0.7615,612,0.8526,0.9432,0.8956,2183
4,strat_y_K,RF,test_data,855,0.4433,0.3348,0.6772,0.5170,0.4957,0.4956,...,0.4969,294,0.2556,0.2698,0.2625,126,0.6604,0.8092,0.7273,435
5,strat_y_K,RF,entire_data,4279,0.8518,0.7509,0.9168,0.7899,0.8050,0.7821,...,0.7644,1484,0.5607,0.8072,0.6618,612,0.8887,0.9542,0.9203,2183
6,strat_y_K,XGB,test_data,855,0.4620,0.3615,0.6971,0.5500,0.5331,0.5346,...,0.5357,294,0.3281,0.3333,0.3307,126,0.6789,0.8069,0.7374,435
7,strat_y_K,XGB,entire_data,4279,0.8600,0.7513,0.9229,0.8288,0.8287,0.8223,...,0.8039,1484,0.7056,0.8186,0.7579,612,0.8613,0.9533,0.9050,2183


## Cell 6 — S/R/P confusion matrices

These matrices visualize the **derived branch decision**, not an independently trained 3-class classifier.


In [6]:
for (s,m,sc),f in predictions.items():
    cm=confusion_matrix(f.true_branch,f.predicted_branch,labels=["S","R","P"])
    fig,ax=plt.subplots(figsize=(6,5)); ConfusionMatrixDisplay(cm,display_labels=["S","R","P"]).plot(ax=ax,values_format="d")
    ax.set_title(f"Derived S/R/P — {m} — {s} — {sc}"); fig.tight_layout()
    fig.savefig(OUTPUT_DIR/f"confusion_SRP_{s}_{m}_{sc}.png",dpi=180,bbox_inches="tight"); plt.show()


C:\Users\n1a9m\AppData\Local\Temp\ipykernel_25012\198175631.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(OUTPUT_DIR/f"confusion_SRP_{s}_{m}_{sc}.png",dpi=180,bbox_inches="tight"); plt.show()
C:\Users\n1a9m\AppData\Local\Temp\ipykernel_25012\198175631.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(OUTPUT_DIR/f"confusion_SRP_{s}_{m}_{sc}.png",dpi=180,bbox_inches="tight"); plt.show()
C:\Users\n1a9m\AppData\Local\Temp\ipykernel_25012\198175631.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(OUTPUT_DIR/f"confusion_SRP_{s}_{m}_{sc}.png",dpi=180,bbox_inches="tight"); plt.show()
C:\Users\n1a9m\AppData\Local\Temp\ipykernel_25012\198175631.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(OUTPUT_DIR/f"confusion_SRP_{s}_{m}_{sc}.png",dpi=180,bbox_inches="tight"); plt.show()
C:\Users\n1a9m\AppData\Local\Temp\ipyker

## Cell 7 — Wrong predictions and important failures

**Existing function reused:** `filter_tolerance_sheet_for_important_failures()`.

Wrong prediction means the selected algorithm is not one of the tied fastest algorithms in the 18-way candidate set.

For the existing important-failure filter we provide:

- `fastest_time` = **global minimum among all 24 algorithms**
- `predicted_time` = runtime of the predicted 18-way algorithm

Thus the failure severity remains directly comparable with the previous 24-way global-runtime analyses.


In [7]:
wrong_frames={}; important_frames={}; summary=[]
for key,f in predictions.items():
    s,m,sc=key; wrong=f[~f.algorithm_correct].copy()
    inp=wrong.copy(); inp["fastest_time"]=inp["global_fastest_time_24"]; inp["predicted_time"]=inp["predicted_runtime"]
    important,stats=filter_tolerance_sheet_for_important_failures(inp)
    wrong_frames[key]=wrong; important_frames[key]=important
    summary.append({"Split":s,"Model":m,"Scope":sc,"Total Predictions":len(f),"Wrong Predictions":len(wrong),
                    "Important Failures":len(important),
                    "Important / Wrong (%)":round(100*len(important)/len(wrong),2) if len(wrong) else 0,
                    "Important / All (%)":round(100*len(important)/len(f),2) if len(f) else 0})
failure_summary=pd.DataFrame(summary); display(failure_summary); failure_summary.to_csv(OUTPUT_DIR/"failure_summary.csv",index=False)


,Split,Model,Scope,Total Predictions,Wrong Predictions,Important Failures,Important / Wrong (%),Important / All (%)
0,strat_y,RF,test_data,856,456,104,22.81,12.15
1,strat_y,RF,entire_data,4279,851,163,19.15,3.81
2,strat_y,XGB,test_data,856,441,97,22.00,11.33
3,strat_y,XGB,entire_data,4279,591,111,18.78,2.59
4,strat_y_K,RF,test_data,855,476,97,20.38,11.35
5,strat_y_K,RF,entire_data,4279,634,109,17.19,2.55
6,strat_y_K,XGB,test_data,855,460,98,21.30,11.46
7,strat_y_K,XGB,entire_data,4279,599,110,18.36,2.57


## Cell 8 — Enrich important failures with the established Top-1…Top-6 layout

A local enrichment helper is required because the models are being run directly in the notebook rather than through a saved failure-analysis workbook.

The logic is the established one:

- preserve descending model-probability order;
- `fastest_algo_rank` = rank of the true 18-way family optimum in that probability ordering;
- for Top-K, choose the minimum-runtime algorithm among the first K predicted candidates;
- **Top-K ratio denominator = global minimum runtime among all 24 algorithms**;
- `origin` = S/R/P branch of the true fastest 18-way algorithm.


In [8]:
FAILURE_COLUMNS=["dataset","k_value","scheme","origin","fastest_algo_rank","fastest_algo_tuple"]
for k in range(1,7): FAILURE_COLUMNS += [f"top{k}_min_time_ratio",f"top{k}_candidate"]
FAILURE_COLUMNS += ["all_predictions_sorted_by_prob"]

def enrich_important(frame):
    rows=[]
    for _,r in frame.iterrows():
        ranking=r["all_predictions_sorted_by_prob"]; true_set=set(r["true_fastest_algorithms_18"])
        # If multiple 18-way algorithms are tied, use the highest-probability true optimum for rank display.
        true_ranked=[(i,z) for i,z in enumerate(ranking,1) if z[1] in true_set]
        rank,best_true_entry=min(true_ranked,key=lambda q:q[0])
        fa=best_true_entry[1]; ft=float(df.loc[int(r.source_row_index),fa])
        rec={"dataset":r.dataset,"k_value":r.k_value,"scheme":"18-class S/R/P flat",
             "origin":ALGO_TO_BRANCH[fa],"fastest_algo_rank":rank,
             "fastest_algo_tuple":f"({ft:.4f}, {fa!r}, {rank})",
             "all_predictions_sorted_by_prob":str(ranking)}
        for k in range(1,7):
            cand=ranking[:k]; best=min(cand,key=lambda z:(z[2],z[1])) if cand else None
            rec[f"top{k}_min_time_ratio"]=round(best[0],4) if best else pd.NA
            rec[f"top{k}_candidate"]=f"({best[2]:.4f}, {best[1]!r}, {best[3]:.4f})" if best else pd.NA
        rows.append(rec)
    return pd.DataFrame(rows,columns=FAILURE_COLUMNS)

enriched={k:enrich_important(v) for k,v in important_frames.items()}
display(enriched[("strat_y","RF","test_data")].head())


,dataset,k_value,scheme,origin,fastest_algo_rank,fastest_algo_tuple,top1_min_time_ratio,top1_candidate,top2_min_time_ratio,top2_candidate,top3_min_time_ratio,top3_candidate,top4_min_time_ratio,top4_candidate,top5_min_time_ratio,top5_candidate,top6_min_time_ratio,top6_candidate,all_predictions_sorted_by_prob
0,wiki-Talk,3,18-class S/R/P flat,P,2,"(37.3333, 'UMKP-oriented-P-bnb', 2)",2.8787,"(107.4704, 'UMKP-P-bnb', 0.5148)",1.0000,"(37.3333, 'UMKP-oriented-P-bnb', 0.2595)",1.0000,"(37.3333, 'UMKP-oriented-P-bnb', 0.2595)",1.0000,"(37.3333, 'UMKP-oriented-P-bnb', 0.2595)",1.0000,"(37.3333, 'UMKP-oriented-P-bnb', 0.2595)",1.0000,"(37.3333, 'UMKP-oriented-P-bnb', 0.2595)","[(2.8786739988160703, 'UMKP-P-bnb', 107.4704, ..."
1,edit-jawiki,6,18-class S/R/P flat,R,12,"(36.2277, 'UMKP-oriented-R-bnb', 12)",1.6261,"(58.9088, 'UMKP+-P-bnb', 0.3984)",1.0746,"(38.9295, 'kPEX-oriented-P-bnb', 0.2747)",1.0746,"(38.9295, 'kPEX-oriented-P-bnb', 0.2747)",1.0746,"(38.9295, 'kPEX-oriented-P-bnb', 0.2747)",1.0746,"(38.9295, 'kPEX-oriented-P-bnb', 0.2747)",1.0746,"(38.9295, 'kPEX-oriented-P-bnb', 0.2747)","[(1.6260706586396598, 'UMKP+-P-bnb', 58.9088, ..."
2,fp,6,18-class S/R/P flat,S,10,"(30.4763, 'UMKP-S-bnb', 10)",1.2903,"(39.3227, 'UMKP+-R-bnb', 0.5532)",1.2903,"(39.3227, 'UMKP+-R-bnb', 0.5532)",1.2903,"(39.3227, 'UMKP+-R-bnb', 0.5532)",1.2903,"(39.3227, 'UMKP+-R-bnb', 0.5532)",1.2903,"(39.3227, 'UMKP+-R-bnb', 0.5532)",1.2903,"(39.3227, 'UMKP+-R-bnb', 0.5532)","[(1.290271456836952, 'UMKP+-R-bnb', 39.3227, 0..."
3,sx-stackoverflow,2,18-class S/R/P flat,P,3,"(22.0867, 'kPEX-oriented-P-bnb', 3)",19.3984,"(356.3742, 'UMKP-oriented-P-bnb', 0.4079)",5.4772,"(100.6242, 'UMKP-P-bnb', 0.2313)",1.2022,"(22.0867, 'kPEX-oriented-P-bnb', 0.1803)",1.2022,"(22.0867, 'kPEX-oriented-P-bnb', 0.1803)",1.2022,"(22.0867, 'kPEX-oriented-P-bnb', 0.1803)",1.2022,"(22.0867, 'kPEX-oriented-P-bnb', 0.1803)","[(19.398420362195377, 'UMKP-oriented-P-bnb', 3..."
4,link-dynamic-frwiki,7,18-class S/R/P flat,S,3,"(0.1338, 'UMKP+-S-bnb', 3)",13452.9148,"(1800.0000, 'UMKP-oriented-S-bnb', 0.6284)",1.8804,"(0.2516, 'kPEX-S-bnb', 0.0999)",1.0000,"(0.1338, 'UMKP+-S-bnb', 0.0809)",1.0000,"(0.1338, 'UMKP+-S-bnb', 0.0809)",1.0000,"(0.1338, 'UMKP+-S-bnb', 0.0809)",1.0000,"(0.1338, 'UMKP+-S-bnb', 0.0809)","[(13452.914798206279, 'UMKP-oriented-S-bnb', 1..."


## Cell 9 — Test/entire important-failure overlap summary

For each model and split, overlap is based on the same `(dataset, k_value)` being an important failure in both the held-out-test and entire-data analyses.


In [9]:
overlap=[]
for s in SPLITS:
    for m in MODELS:
        te=enriched[(s,m,"test_data")]; en=enriched[(s,m,"entire_data")]
        kt=set(zip(te.dataset.astype(str),te.k_value)) if len(te) else set()
        ke=set(zip(en.dataset.astype(str),en.k_value)) if len(en) else set()
        shared=kt&ke
        overlap.append({"Split":s,"Model":m,"Test Important Failures":len(te),"Entire Important Failures":len(en),
                        "Overlapping Dataset/K Cases":len(shared),"Test Only":len(kt-ke),"Entire Only":len(ke-kt)})
overlap_summary=pd.DataFrame(overlap); display(overlap_summary)


,Split,Model,Test Important Failures,Entire Important Failures,Overlapping Dataset/K Cases,Test Only,Entire Only
0,strat_y,RF,104,163,104,0,59
1,strat_y,XGB,97,111,97,0,14
2,strat_y_K,RF,97,109,97,0,12
3,strat_y_K,XGB,98,110,98,0,12


## Cell 10 — Write the important-failure workbook

The workbook contains:

- `summary` — counts plus test/entire overlap by model and split.
- **8 detail sheets** — RF/XGB × `strat_y`/`strat_y_K` × test/entire.

Every detail sheet has exactly the requested established failure-analysis columns.


In [10]:
summary_sheet=failure_summary.merge(overlap_summary,on=["Split","Model"],how="left")
book=OUTPUT_DIR/"important_failure_analysis.xlsx"
with pd.ExcelWriter(book,engine="openpyxl") as w:
    summary_sheet.to_excel(w,sheet_name="summary",index=False)
    for (s,m,sc),e in enriched.items():
        e[FAILURE_COLUMNS].to_excel(w,sheet_name=f"{s}_{m}_{'test' if sc=='test_data' else 'entire'}"[:31],index=False)
print("Wrote:",book)
display(summary_sheet)


Wrote: restructured_pipeline\outputs\reports\SRP_18class_flat\important_failure_analysis.xlsx


,Split,Model,Scope,Total Predictions,Wrong Predictions,Important Failures,Important / Wrong (%),Important / All (%),Test Important Failures,Entire Important Failures,Overlapping Dataset/K Cases,Test Only,Entire Only
0,strat_y,RF,test_data,856,456,104,22.81,12.15,104,163,104,0,59
1,strat_y,RF,entire_data,4279,851,163,19.15,3.81,104,163,104,0,59
2,strat_y,XGB,test_data,856,441,97,22.00,11.33,97,111,97,0,14
3,strat_y,XGB,entire_data,4279,591,111,18.78,2.59,97,111,97,0,14
4,strat_y_K,RF,test_data,855,476,97,20.38,11.35,97,109,97,0,12
5,strat_y_K,RF,entire_data,4279,634,109,17.19,2.55,97,109,97,0,12
6,strat_y_K,XGB,test_data,855,460,98,21.30,11.46,98,110,98,0,12
7,strat_y_K,XGB,entire_data,4279,599,110,18.36,2.57,98,110,98,0,12


## Cell 11 — Slide-ready tables

The first table reports held-out algorithm and derived S/R/P performance. The second reports held-out wrong/important failures. The third reports test/entire important-failure overlap.


In [11]:
slide_metrics=metrics[metrics.Scope.eq("test_data")].copy()
pct=["Algorithm Accuracy","Algorithm Weighted F1","S/R/P Accuracy","S/R/P Mean F1","S/R/P Weighted F1",
     "S Precision","S Recall","S F1","R Precision","R Recall","R F1","P Precision","P Recall","P F1"]
for c in pct: slide_metrics[c]=(100*slide_metrics[c]).round(2)
slide_metrics=slide_metrics[["Split","Model","Total Cases",*pct]]
slide_fail=failure_summary[failure_summary.Scope.eq("test_data")].copy()
display(slide_metrics); display(slide_fail); display(overlap_summary)
slide_metrics.to_csv(OUTPUT_DIR/"slide_metrics.csv",index=False)
slide_fail.to_csv(OUTPUT_DIR/"slide_failures.csv",index=False)
overlap_summary.to_csv(OUTPUT_DIR/"slide_failure_overlap.csv",index=False)


,Split,Model,Total Cases,Algorithm Accuracy,Algorithm Weighted F1,S/R/P Accuracy,S/R/P Mean F1,S/R/P Weighted F1,S Precision,S Recall,S F1,R Precision,R Recall,R F1,P Precision,P Recall,P F1
0,strat_y,RF,856,46.73,36.24,70.21,53.10,61.08,62.75,43.10,51.10,28.66,36.59,32.14,71.52,81.19,76.05
2,strat_y,XGB,856,48.48,38.83,70.33,55.03,61.98,63.93,47.14,54.26,34.04,39.02,36.36,69.96,79.59,74.46
4,strat_y_K,RF,855,44.33,33.48,67.72,49.56,57.96,63.49,40.82,49.69,25.56,26.98,26.25,66.04,80.92,72.73
6,strat_y_K,XGB,855,46.20,36.15,69.71,53.46,60.81,64.29,45.92,53.57,32.81,33.33,33.07,67.89,80.69,73.74


,Split,Model,Scope,Total Predictions,Wrong Predictions,Important Failures,Important / Wrong (%),Important / All (%)
0,strat_y,RF,test_data,856,456,104,22.81,12.15
2,strat_y,XGB,test_data,856,441,97,22.00,11.33
4,strat_y_K,RF,test_data,855,476,97,20.38,11.35
6,strat_y_K,XGB,test_data,855,460,98,21.30,11.46


,Split,Model,Test Important Failures,Entire Important Failures,Overlapping Dataset/K Cases,Test Only,Entire Only
0,strat_y,RF,104,163,104,0,59
1,strat_y,XGB,97,111,97,0,14
2,strat_y_K,RF,97,109,97,0,12
3,strat_y_K,XGB,98,110,98,0,12


# Cell-by-cell reuse summary

| Cell | Purpose | Existing pipeline functionality reused |
|---|---|---|
| 1 | Define 18-class S/R/P experiment | Canonical algorithm groups from `features.py`; configuration via `load_yaml_config()` |
| 2 | Load/preprocess data | **`train.py -> load_and_preprocess_data()`**, which internally performs the canonical loading, weighting, k-core statistics, post-processing and feature sanitization |
| 3 | Split, tie-expand training, train RF/XGB | **`fit_random_forest_classifier()` and `fit_xgb_classifier()`**; these call the existing flat split functions with `algorithm_cols=SRP_ALGOS` |
| 4 | Generate 18-way rankings | Small notebook helper because models are kept in memory rather than saved/reloaded |
| 5 | Algorithm + derived S/R/P metrics | sklearn reporting only; predictions remain 18-class |
| 6 | Derived S/R/P confusion matrices | sklearn visualization |
| 7 | Wrong + important failures | **existing `filter_tolerance_sheet_for_important_failures()`** |
| 8 | Top-1…Top-6 enrichment | Notebook helper matching the established failure-analysis semantics; denominator is global 24-way minimum |
| 9 | Test/entire overlap | Simple `(dataset,K)` set comparison |
| 10 | Excel workbook | Eight requested detail sheets + summary |
| 11 | Slide tables | Presentation-only formatting |

## Key change from the previous notebook

The previous notebook trained:

`features -> S/R/P`

This notebook trains:

`features -> one of 18 algorithms -> derive S/R/P from algorithm name`

Therefore the model genuinely provides probabilities for all 18 algorithms, making `fastest_algo_rank`, Top-1…Top-6 candidates/ratios, and `all_predictions_sorted_by_prob` meaningful rather than synthetic.
